# Análisis de customers.parquet

**Autor:** Daniel Guzmán  
**Fecha:** 2026-04-23  
**Entorno:** Databricks

In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, when

In [0]:
catalog = "workspace"
schema = "default"
volume = "customers_files_daniel"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

print(path_volume)

In [0]:
inicio = time.time()

df_parquet = spark.read.parquet(f"{path_volume}/customers.parquet")

total_registros = df_parquet.count()
fin = time.time()

print(f"Tiempo de lectura: {fin - inicio:.4f} segundos")
print(f"Registros: {total_registros}")
print(f"Columnas: {df_parquet.columns}")

In [0]:
display(df_parquet.limit(5))

In [0]:
print("Tipos de datos:")
for col_name, dtype in df_parquet.dtypes:
    print(f"{col_name}: {dtype}")

In [0]:
print("Shape:")
print(f"Filas: {df_parquet.count()}")
print(f"Columnas: {len(df_parquet.columns)}")

In [0]:
df_parquet.printSchema()

In [0]:
nulos_df = df_parquet.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_parquet.columns
])

display(nulos_df)

In [0]:
for c in df_parquet.columns:
    print(f"{c}: {df_parquet.select(c).distinct().count()} valores únicos")

In [0]:
display(df_parquet.describe())

In [0]:
clientes_por_pais = (
    df_parquet.groupBy("Country")
    .count()
    .withColumnRenamed("count", "total_clientes")
    .orderBy(F.col("total_clientes").desc())
)

display(clientes_por_pais)

In [0]:
display(clientes_por_pais.limit(5))

In [0]:
empresas_distintas = df_parquet.select("Company").distinct().count()
print(f"Empresas distintas: {empresas_distintas}")

In [0]:
nombres_frecuentes = (
    df_parquet.withColumn(
        "Nombre_Completo",
        F.concat_ws(" ", F.col("First_Name"), F.col("Last_Name"))
    )
    .groupBy("Nombre_Completo")
    .count()
    .orderBy(F.col("count").desc())
)

display(nombres_frecuentes.limit(1))

In [0]:
clientes_sin_ciudad = df_parquet.filter(
    F.col("City").isNull() | (F.trim(F.col("City")) == "")
).count()

print(f"Clientes sin ciudad registrada: {clientes_sin_ciudad}")

## Reflexión sobre el formato .parquet

- Tiempo de lectura registrado:  1.0693 segundos
- Tamaño del archivo: 1.06 MB aprox.
- Fue fácil de leer con PySpark en Databricks.
- Como ventaja, Parquet es compacto, columnar y muy eficiente para análisis. Como desventaja, no es tan legible para humanos como CSV o JSON.
- Usaría Parquet en pipelines analíticos, data lakes y procesos donde importe el rendimiento de lectura y almacenamiento.